In [3]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
import joblib   
import os

In [7]:
INPUT_CSV = "../data/raw/Banglore_traffic_Dataset.csv"   
OUTPUT_PROCESSED_CSV = "bangalore_traffic_processed.csv"
OUTPUT_SCALED_CSV = "bangalore_traffic_scaled.csv"
SCALER_PATH = "traffic_scaler.pkl"

In [10]:
df = pd.read_csv(INPUT_CSV)

print("Raw shape:", df.shape)
df.head()

Raw shape: (8936, 16)


,Date,Area Name,Road/Intersection Name,Traffic Volume,Average Speed,Travel Time Index,Congestion Level,Road Capacity Utilization,Incident Reports,Environmental Impact,Public Transport Usage,Traffic Signal Compliance,Parking Usage,Pedestrian and Cyclist Count,Weather Conditions,Roadwork and Construction Activity
0,2022-01-01,Indiranagar,100 Feet Road,50590,50.230299,1.500000,100.000000,100.000000,0,151.180,70.632330,84.044600,85.403629,111,Clear,No
1,2022-01-01,Indiranagar,CMH Road,30825,29.377125,1.500000,100.000000,100.000000,1,111.650,41.924899,91.407038,59.983689,100,Clear,No
2,2022-01-01,Whitefield,Marathahalli Bridge,7399,54.474398,1.039069,28.347994,36.396525,0,64.798,44.662384,61.375541,95.466020,189,Clear,No
3,2022-01-01,Koramangala,Sony World Junction,60874,43.817610,1.500000,100.000000,100.000000,1,171.748,32.773123,75.547092,63.567452,111,Clear,No
4,2022-01-01,Koramangala,Sarjapur Road,57292,41.116763,1.500000,100.000000,100.000000,3,164.584,35.092601,64.634762,93.155171,104,Clear,No


In [11]:
df = df.drop_duplicates()
print("After dropping duplicates:", df.shape)

After dropping duplicates: (8936, 16)


In [13]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8936 entries, 0 to 8935
Data columns (total 18 columns):
 #   Column                              Non-Null Count  Dtype         
---  ------                              --------------  -----         
 0   Date                                8936 non-null   datetime64[ns]
 1   Area Name                           8936 non-null   object        
 2   Road/Intersection Name              8936 non-null   object        
 3   Traffic Volume                      8936 non-null   int64         
 4   Average Speed                       8936 non-null   float64       
 5   Travel Time Index                   8936 non-null   float64       
 6   Congestion Level                    8936 non-null   float64       
 7   Road Capacity Utilization           8936 non-null   float64       
 8   Incident Reports                    8936 non-null   int64         
 9   Environmental Impact                8936 non-null   float64       
 10  Public Transport Usage  

In [12]:
if 'Date' in df.columns:
    df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
    df['DayOfWeek'] = df['Date'].dt.dayofweek
    df['IsWeekend'] = df['DayOfWeek'].apply(lambda x: 1 if x >= 5 else 0)
else:
    print("⚠️ Date column not found, skipping date-based features.")
    df['IsWeekend'] = 0

df.head()


,Date,Area Name,Road/Intersection Name,Traffic Volume,Average Speed,Travel Time Index,Congestion Level,Road Capacity Utilization,Incident Reports,Environmental Impact,Public Transport Usage,Traffic Signal Compliance,Parking Usage,Pedestrian and Cyclist Count,Weather Conditions,Roadwork and Construction Activity,DayOfWeek,IsWeekend
0,2022-01-01,Indiranagar,100 Feet Road,50590,50.230299,1.500000,100.000000,100.000000,0,151.180,70.632330,84.044600,85.403629,111,Clear,No,5,1
1,2022-01-01,Indiranagar,CMH Road,30825,29.377125,1.500000,100.000000,100.000000,1,111.650,41.924899,91.407038,59.983689,100,Clear,No,5,1
2,2022-01-01,Whitefield,Marathahalli Bridge,7399,54.474398,1.039069,28.347994,36.396525,0,64.798,44.662384,61.375541,95.466020,189,Clear,No,5,1
3,2022-01-01,Koramangala,Sony World Junction,60874,43.817610,1.500000,100.000000,100.000000,1,171.748,32.773123,75.547092,63.567452,111,Clear,No,5,1
4,2022-01-01,Koramangala,Sarjapur Road,57292,41.116763,1.500000,100.000000,100.000000,3,164.584,35.092601,64.634762,93.155171,104,Clear,No,5,1


In [14]:
cat_cols = []

if 'Weather Conditions' in df.columns:
    df['Weather Conditions'] = df['Weather Conditions'].astype('category')
    cat_cols.append('Weather Conditions')

if 'Roadwork and Construction Activity' in df.columns:
    df['Roadwork and Construction Activity'] = df['Roadwork and Construction Activity'].astype('category')
    cat_cols.append('Roadwork and Construction Activity')

print("Categorical columns detected:", cat_cols)


Categorical columns detected: ['Weather Conditions', 'Roadwork and Construction Activity']


In [15]:
if cat_cols:
    df = pd.get_dummies(df, columns=cat_cols, drop_first=True)

print("After encoding:", df.shape)
df.head()

After encoding: (8936, 21)


,Date,Area Name,Road/Intersection Name,Traffic Volume,Average Speed,Travel Time Index,Congestion Level,Road Capacity Utilization,Incident Reports,Environmental Impact,...,Traffic Signal Compliance,Parking Usage,Pedestrian and Cyclist Count,DayOfWeek,IsWeekend,Weather Conditions_Fog,Weather Conditions_Overcast,Weather Conditions_Rain,Weather Conditions_Windy,Roadwork and Construction Activity_Yes
0,2022-01-01,Indiranagar,100 Feet Road,50590,50.230299,1.500000,100.000000,100.000000,0,151.180,...,84.044600,85.403629,111,5,1,False,False,False,False,False
1,2022-01-01,Indiranagar,CMH Road,30825,29.377125,1.500000,100.000000,100.000000,1,111.650,...,91.407038,59.983689,100,5,1,False,False,False,False,False
2,2022-01-01,Whitefield,Marathahalli Bridge,7399,54.474398,1.039069,28.347994,36.396525,0,64.798,...,61.375541,95.466020,189,5,1,False,False,False,False,False
3,2022-01-01,Koramangala,Sony World Junction,60874,43.817610,1.500000,100.000000,100.000000,1,171.748,...,75.547092,63.567452,111,5,1,False,False,False,False,False
4,2022-01-01,Koramangala,Sarjapur Road,57292,41.116763,1.500000,100.000000,100.000000,3,164.584,...,64.634762,93.155171,104,5,1,False,False,False,False,False


In [16]:
base_features = [
    'Traffic Volume',
    'Average Speed',
    'Travel Time Index',
    'Congestion Level',
    'Road Capacity Utilization',
    'Incident Reports',
    'Environmental Impact',
    'Pedestrian and Cyclist Count',
    'IsWeekend'
]

# Keep only columns that exist
selected_features = [col for col in base_features if col in df.columns]

# Add encoded columns
encoded_cols = [
    col for col in df.columns
    if col.startswith('Weather Conditions_')
    or col.startswith('Roadwork and Construction Activity_')
]

selected_features += encoded_cols

print("📌 Features selected for clustering:\n")
for f in selected_features:
    print(" -", f)



📌 Features selected for clustering:

 - Traffic Volume
 - Average Speed
 - Travel Time Index
 - Congestion Level
 - Road Capacity Utilization
 - Incident Reports
 - Environmental Impact
 - Pedestrian and Cyclist Count
 - IsWeekend
 - Weather Conditions_Fog
 - Weather Conditions_Overcast
 - Weather Conditions_Rain
 - Weather Conditions_Windy
 - Roadwork and Construction Activity_Yes


In [17]:
df_features = df[selected_features].copy()

print("Final feature dataset shape:", df_features.shape)
df_features.head()


Final feature dataset shape: (8936, 14)


,Traffic Volume,Average Speed,Travel Time Index,Congestion Level,Road Capacity Utilization,Incident Reports,Environmental Impact,Pedestrian and Cyclist Count,IsWeekend,Weather Conditions_Fog,Weather Conditions_Overcast,Weather Conditions_Rain,Weather Conditions_Windy,Roadwork and Construction Activity_Yes
0,50590,50.230299,1.500000,100.000000,100.000000,0,151.180,111,1,False,False,False,False,False
1,30825,29.377125,1.500000,100.000000,100.000000,1,111.650,100,1,False,False,False,False,False
2,7399,54.474398,1.039069,28.347994,36.396525,0,64.798,189,1,False,False,False,False,False
3,60874,43.817610,1.500000,100.000000,100.000000,1,171.748,111,1,False,False,False,False,False
4,57292,41.116763,1.500000,100.000000,100.000000,3,164.584,104,1,False,False,False,False,False


In [18]:
df_features = df_features.astype(float)
df_features = df_features.fillna(df_features.mean())

df_features.head()


,Traffic Volume,Average Speed,Travel Time Index,Congestion Level,Road Capacity Utilization,Incident Reports,Environmental Impact,Pedestrian and Cyclist Count,IsWeekend,Weather Conditions_Fog,Weather Conditions_Overcast,Weather Conditions_Rain,Weather Conditions_Windy,Roadwork and Construction Activity_Yes
0,50590.0,50.230299,1.500000,100.000000,100.000000,0.0,151.180,111.0,1.0,0.0,0.0,0.0,0.0,0.0
1,30825.0,29.377125,1.500000,100.000000,100.000000,1.0,111.650,100.0,1.0,0.0,0.0,0.0,0.0,0.0
2,7399.0,54.474398,1.039069,28.347994,36.396525,0.0,64.798,189.0,1.0,0.0,0.0,0.0,0.0,0.0
3,60874.0,43.817610,1.500000,100.000000,100.000000,1.0,171.748,111.0,1.0,0.0,0.0,0.0,0.0,0.0
4,57292.0,41.116763,1.500000,100.000000,100.000000,3.0,164.584,104.0,1.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
# Scale the feature 

scaler = StandardScaler()
scaled_array = scaler.fit_transform(df_features)

df_scaled = pd.DataFrame(scaled_array, columns=df_features.columns)
df_scaled.head()


,Traffic Volume,Average Speed,Travel Time Index,Congestion Level,Road Capacity Utilization,Incident Reports,Environmental Impact,Pedestrian and Cyclist Count,IsWeekend,Weather Conditions_Fog,Weather Conditions_Overcast,Weather Conditions_Rain,Weather Conditions_Windy,Roadwork and Construction Activity_Yes
0,1.642475,1.007120,0.752807,0.815148,0.480677,-1.105933,1.642475,-0.095987,1.577308,-0.346728,-0.411866,-0.319352,-0.224289,-0.330924
1,0.122217,-0.940566,0.752807,0.815148,0.480677,-0.401692,0.122217,-0.394815,1.577308,-0.346728,-0.411866,-0.319352,-0.224289,-0.330924
2,-1.679633,1.403518,-2.035479,-2.229745,-3.354921,-1.105933,-1.679633,2.022972,1.577308,-0.346728,-0.411866,-0.319352,-0.224289,-0.330924
3,2.433486,0.408175,0.752807,0.815148,0.480677,-0.401692,2.433486,-0.095987,1.577308,-0.346728,-0.411866,-0.319352,-0.224289,-0.330924
4,2.157971,0.155916,0.752807,0.815148,0.480677,1.006791,2.157971,-0.286151,1.577308,-0.346728,-0.411866,-0.319352,-0.224289,-0.330924


In [20]:
df_scaled.shape

(8936, 14)

In [21]:
save_path = "../data/processed"

df_features.to_csv(f"{save_path}\\bangalore_traffic_processed.csv", index=False)
df_scaled.to_csv(f"{save_path}\\bangalore_traffic_scaled.csv", index=False)
joblib.dump(scaler, f"{save_path}\\traffic_scaler.pkl")

print("✔ Saved:", f"{save_path}\\bangalore_traffic_processed.csv")
print("✔ Saved:", f"{save_path}\\bangalore_traffic_scaled.csv")
print("✔ Saved scaler:", f"{save_path}\\traffic_scaler.pkl")

✔ Saved: ../data/processed\bangalore_traffic_processed.csv
✔ Saved: ../data/processed\bangalore_traffic_scaled.csv
✔ Saved scaler: ../data/processed\traffic_scaler.pkl
